### Week 36

In [ ]:
import pandas as pd

In [ ]:
validation_set = pd.read_parquet("data/validation.parquet")
train_set = pd.read_parquet("data/train.parquet")

In [ ]:
validation_set.shape

(3011, 7)

In [ ]:
train_set.shape

(15343, 7)

In [ ]:
train_set.columns


Index(['question', 'context', 'lang', 'answerable', 'answer_start', 'answer',
       'answer_inlang'],
      dtype='object')

In [ ]:
strip_punctuation = lambda line: [word.strip(string.punctuation+"؟")for word in line.split(" ")]
count_words = lambda line: len(line)

In [ ]:
import torch
print("Torch:", torch.__version__)        # should show +cu126
print("CUDA runtime:", torch.version.cuda) # '12.6'
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

  I wanted to strip punctuation, but certain languages that we do not include in the analysis have special punctuation that has to be included in thw stripping pool

### Week 36 - rule-based classifier


In [3]:
import pandas as pd
import string
import numpy as np

In [4]:
val_tr = pd.read_csv("data/validation_translated.csv")
train_tr = pd.read_csv("data/train_translated.csv")

In [5]:
val_tr.shape

(1155, 13)

In [6]:
train_tr.shape

(6335, 13)

In [7]:
train_tr

,Unnamed: 0,question,context,lang,answerable,answer_start,answer,answer_inlang,question_stripped,question_wordcount,context_stripped,context_wordcount,question_translated
0,4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,NaN,"['30년', '전쟁의', '승자는', '누구인가']",4,"['The', 'conflict', 'between', 'France', 'and'...",108,Who is the winner of the Thirty Years' War?
1,4793,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,NaN,"['엑스선은', '누가', '발견하였는가']",3,"['X-rays', 'make', 'up', 'X-radiation', 'a', '...",122,Who discovered X-rays?
2,4794,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever ci...",ko,True,188,2004,NaN,"['아테네에서', '언제', '가장', '최근의', '올림픽이', '올렸나요']",6,"['In', '2022', 'Beijing', 'will', 'become', 't...",197,When was the last Olympic Games held in Athens?
3,4795,세상에서 가장 오래된 방송사는 무엇인가?,The British Broadcasting Corporation (BBC) is ...,ko,True,4,British Broadcasting Corporation (BBC),NaN,"['세상에서', '가장', '오래된', '방송사는', '무엇인가']",5,"['The', 'British', 'Broadcasting', 'Corporatio...",70,What's the oldest broadcaster in the world?
4,4796,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palest...",ko,True,205,Jerusalem,NaN,"['팔레스타인', '수도는', '어딘가요']",3,"['Palestine', '', '', 'officially', 'the', 'St...",84,Where's the Palestinian capital?
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6330,15338,소말리아는 2차 개헌을 언제 했나요?,"In February 2012, Somali government officials ...",ko,True,923,23 June 2012,NaN,"['소말리아는', '2차', '개헌을', '언제', '했나요']",5,"['In', 'February', '2012', 'Somali', 'governme...",181,When did Somalia make its second inauguration?
6331,15339,세상에서 가장 먼저 시작된 교통수단은 무엇인가?,The first earth tracks were created by humans ...,ko,True,160,animals,NaN,"['세상에서', '가장', '먼저', '시작된', '교통수단은', '무엇인가']",6,"['The', 'first', 'earth', 'tracks', 'were', 'c...",118,What was the world's first transportation system?
6332,15340,2019년 이집트의 지도자는 누구인가?,"Abdel Fattah Saeed Hussein Khalil El-Sisi ( """"...",ko,True,0,Abdel Fattah Saeed Hussein Khalil El-Sisi,NaN,"['2019년', '이집트의', '지도자는', '누구인가']",4,"['Abdel', 'Fattah', 'Saeed', 'Hussein', 'Khali...",30,Who is Egypt's leader in 2019?
6333,15341,독일에서 가장 인구밀도가 높은 도시는 무엇인가?,Munich (; ; ) is the capital and most populous...,ko,True,205,Berlin,NaN,"['독일에서', '가장', '인구밀도가', '높은', '도시는', '무엇인가']",6,"['Munich', '', '', '', 'is', 'the', 'capital',...",118,What is the most densely populated city in Ger...


In [8]:
strip_first = lambda x: x.split()[0]

In [9]:
import re

question_contractions = {
    "what's": "what is",
    "where's": "where is",
    "who's": "who is",
    "when's": "when is",
    "why's": "why is",
    "how's": "how is",
    "in what year": "when",
    "in what month": "when",
    "in what day": "when",
    "in what hour": "when",
    "in what minute": "when",
    "in what second": "when",
    "what're": "what are",
    "where're": "where are",
    "who're": "who are",
    "when're": "when are",
    "why're": "why are",
    "how're": "how are",
    "what've": "what have",
    "where've": "where have",
    "who've": "who have",
    "when've": "when have",
    "why've": "why have",
    "how've": "how have",
    "in which country": "where",
    "in which city": "where",
    "in which state": "where",
    "in which region": "where",
    "in which department": "where",
    "in what country": "where",
    "in what city": "where",
    "in what state": "where",
    "in what region": "where",
    "in what department": "where",
    "in what company": "where",
    "in what years": "when",
    "in what months": "when",
    "in what days": "when",
    "in what hours": "when",
    "in what minutes": "when",
    "in what seconds": "when",
    "in which year": "when",
    "in which month": "when",
    "in which day": "when",
    "in which hour": "when",
    "in which minute": "when",
    "in which second": "when",
    "in which direction": "where",
    "on what day": "when",
    "on what date": "when",
    "on what time": "when",

}

def expand_contractions(text, contractions=question_contractions):
    pattern = re.compile(r'\b(' + '|'.join(re.escape(k) for k in contractions.keys()) + r')\b', flags=re.IGNORECASE)
    return pattern.sub(lambda x: contractions[x.group().lower()], text)


In [10]:
train_tr["question_start"] = train_tr["question_translated"].apply(expand_contractions).apply(strip_first).apply(str.lower)
val_tr["question_start"] = val_tr["question_translated"].apply(expand_contractions).apply(strip_first).apply(str.lower)

#### Rule-based

In [11]:
def top_10(df):
    counts = (
        df.groupby(["answerable", "question_start"])
        .size()
        .rename("count")
        .reset_index()
    )

    counts["answerable"] = counts["answerable"].astype(bool)

    top10 = (
        counts.sort_values(["answerable", "count"], ascending=[True, False])\
              .groupby("answerable", group_keys=False)\
              .apply(lambda x: x) \
              .reset_index(drop=True) # Convert back to DataFrame
    )

    opposite_counts = counts.copy()
    opposite_counts["answerable"] = ~opposite_counts["answerable"]
    opposite_counts = opposite_counts.rename(columns={"count": "count_in_opposite_class"})

    top10_with_opposite = (
        top10.merge(
            opposite_counts[["answerable", "question_start", "count_in_opposite_class"]],
            on=["answerable", "question_start"],
            how="left",
        )
        .fillna({"count_in_opposite_class": 0})
    )
    top10_with_opposite["count_in_opposite_class"] = (
        top10_with_opposite["count_in_opposite_class"].astype(int)
    )

    top10_with_opposite["valid"] = np.select(
        [
            top10_with_opposite["count"] > top10_with_opposite["count_in_opposite_class"],
            top10_with_opposite["count"] < top10_with_opposite["count_in_opposite_class"],
        ],
        [True, False],
        default="tie",
    )
    return top10_with_opposite
# top10_with_opposite["is_more_in_answerable"] = np.where(
#     top10_with_opposite["count"] == top10_with_opposite["count_in_opposite_class"],
#     np.nan,
#     top10_with_opposite["count"] > top10_with_opposite["count_in_opposite_class"],
# )



In [12]:
valid = top_10(val_tr)
train = top_10(train_tr)

C:\Users\dnedi\AppData\Local\Temp\ipykernel_11072\1919682493.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  counts.sort_values(["answerable", "count"], ascending=[True, False])\
C:\Users\dnedi\AppData\Local\Temp\ipykernel_11072\1919682493.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  counts.sort_values(["answerable", "count"], ascending=[True, False])\


In [13]:
train[train["answerable"]==False].head(6).to_latex()

'\\begin{tabular}{lrlrrl}\n\\toprule\n & answerable & question_start & count & count_in_opposite_class & valid \\\\\n\\midrule\n0 & False & is & 139 & 61 & True \\\\\n1 & False & can & 37 & 7 & True \\\\\n2 & False & does & 37 & 25 & True \\\\\n3 & False & what & 29 & 1946 & False \\\\\n4 & False & are & 22 & 9 & True \\\\\n5 & False & do & 17 & 6 & True \\\\\n\\bottomrule\n\\end{tabular}\n'

In [14]:
train[train["answerable"]==True].head().to_latex()

'\\begin{tabular}{lrlrrl}\n\\toprule\n & answerable & question_start & count & count_in_opposite_class & valid \\\\\n\\midrule\n24 & True & what & 1946 & 29 & True \\\\\n25 & True & when & 1247 & 4 & True \\\\\n26 & True & who & 1050 & 10 & True \\\\\n27 & True & how & 787 & 7 & True \\\\\n28 & True & where & 516 & 4 & True \\\\\n\\bottomrule\n\\end{tabular}\n'

In [15]:
train_start_nonans = train[(train["valid"] == "True") & (train["answerable"] == False)]["question_start"].tolist()

In [16]:
train_start_nonans

['is',
 'can',
 'does',
 'are',
 'do',
 'has',
 'did',
 'was',
 'could',
 'at',
 'have',
 'were',
 'jack',
 'major,']

In [17]:
def rule_based(df, non_questions):
  df["prediction_answarable"] = np.where(df["question_start"].isin(non_questions), False, True)
  return df

In [18]:
train_pred = rule_based(train_tr, train_start_nonans)
val_pred = rule_based(val_tr, train_start_nonans)

In [19]:
train_pred["accurate"] = train_pred["answerable"] == train_pred["prediction_answarable"]
val_pred["accurate"] = val_pred["answerable"] == val_pred["prediction_answarable"]

In [23]:
def metrics_with_bacc(y_true, y_pred, name="set"):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    # Confusion matrix components (binary, positive class = 1)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())

    # Safe div helper
    div = lambda a, b: (a / b) if b else 0.0

    accuracy  = div(tp + tn, tp + tn + fp + fn)
    precision = div(tp, tp + fp)
    recall    = div(tp, tp + fn)
    f1        = div(2 * precision * recall, precision + recall)

    tpr_pos = recall
    tpr_neg = div(tn, tn + fp)
    balanced_accuracy = 0.5 * (tpr_pos + tpr_neg)

    print(f"{name} accuracy: {accuracy:.4f},f1: {f1:.4f},baac: {balanced_accuracy:.4f}")
    # print(f"{name} precision: {precision:.4f}")
    # print(f"{name} recall: {recall:.4f}")

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "balanced_accuracy": balanced_accuracy,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }

def accuracy_and_f1(series, name = "sets"):
    accuracy = series.mean()
    tp = series.sum()
    fp = (series == False).sum()
    fn = (series == True).sum() - tp
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 * precision * recall / (precision + recall)
    print(f"{name} accuracy: {accuracy:.4f}, f1: {f1:.4f}")
    #print(f"{name} precision: {precision:.4f}")
    #print(f"{name} recall: {recall:.4f}")


In [24]:
LANG = {"ar": "Arabic", "ko": "Korean", "te": "Telugu"}

In [ ]:
for l in LANG.keys():
    metrics_with_bacc(train_pred[train_pred["lang"]==l]["answerable"],train_pred[train_pred["lang"]==l]["prediction_answarable"], f"Train data {LANG.get(l)}")
    metrics_with_bacc(val_pred[val_pred["lang"]==l]["answerable"],val_pred[val_pred["lang"]==l]["prediction_answarable"], f"Validation data {LANG.get(l)}")

metrics_with_bacc(train_pred["answerable"],train_pred["prediction_answarable"], "Train data")
metrics_with_bacc(val_pred["answerable"],val_pred["prediction_answarable"], "Validation data")

Train data Arabic accuracy: 0.9644,f1: 0.9799,baac: 0.9646
Validation data Arabic accuracy: 0.9807,f1: 0.9889,baac: 0.9725
Train data Korean accuracy: 0.9756,f1: 0.9874,baac: 0.9180
Validation data Korean accuracy: 0.9635,f1: 0.9805,baac: 0.9062
Train data Telugu accuracy: 0.9675,f1: 0.9835,baac: 0.5218
Validation data Telugu accuracy: 0.7500,f1: 0.8567,baac: 0.4985
Train data accuracy: 0.9694,f1: 0.9837,baac: 0.9048
Validation data accuracy: 0.8987,f1: 0.9432,baac: 0.6942


{'accuracy': 0.8987012987012987,
 'precision': 0.9091760299625468,
 'recall': 0.9798183652875883,
 'f1': 0.9431762991743563,
 'balanced_accuracy': 0.6941774753267209,
 'tp': 971,
 'tn': 67,
 'fp': 97,
 'fn': 20}

In [85]:
valid[(valid["valid"] == "True") & (valid["answerable"] == False)]

,answerable,question_start,count,count_in_opposite_class,valid
1,False,is,29,12,True
4,False,can,9,0,True
5,False,does,9,1,True
6,False,are,7,1,True
9,False,"bharat,",5,0,True
10,False,do,5,0,True
13,False,as,3,2,True
14,False,has,3,0,True
16,False,was,2,0,True


In [65]:
valid[valid["valid"]=="True"]

,answerable,question_start,count,count_in_opposite_class,valid
1,False,Is,29,12,True
4,False,Can,9,0,True
5,False,Does,9,1,True
6,False,Are,7,1,True
9,False,"Bharat,",5,0,True
10,False,Do,5,0,True
13,False,As,3,2,True
14,False,Has,3,0,True
16,False,Was,2,0,True
19,True,What,304,35,True
